# Classical shadows (random-Pauli)

Run **one** state-agnostic randomized-measurement campaign, then estimate
**many** observables from that single dataset — the cost of each estimate is
set by its *shadow norm*, not by the number of observables or qubits.

**When to use.** For a *single, fixed* Hamiltonian, `PauliAveraging` is more
shot-efficient — reach for shadows when you want many/unknown local
observables from one campaign, or want to reuse a saved campaign later.


In [ ]:
import numpy as np
from qarp.algorithms import PauliShadow, PauliAveraging
from qarp.algorithms import ShadowDataset
from qarp.blocks import SimpleBlock
from qarp.engines import QarpEngine
from qarp.operators import QubitOperator
import qarp

# Fully seeded on BOTH RNGs so the notebook is reproducible:
#   - PauliShadow(seed=...) fixes the measurement-setting sequence,
#   - QarpEngine(seed=...) fixes the shot sampling.
SETTING_SEED, ENGINE_SEED = 0, 1


## Collect once

Prepare a state (here a 4-qubit GHZ), then collect a random-Pauli campaign.
The primitive is an ordinary `PrimitiveAlgorithm`: `engine.run` returns the
bound operator's expectation, and the raw snapshots are kept on `shadow.dataset`.


In [ ]:
n = 4
ket = SimpleBlock(n)
ket.h(0)
for q in range(n - 1):
    ket.cx(q, q + 1)          # GHZ: (|0000> + |1111>)/sqrt(2)

shadow = PauliShadow(QubitOperator('Z0 Z1'), ket, n_settings=8000, seed=SETTING_SEED)
engine = QarpEngine(seed=ENGINE_SEED)
engine.build([shadow])
zz = engine.run()[0]
print(f'<Z0 Z1> = {zz:.3f}   (exact: 1)')


## Estimate many observables from the *same* dataset

No re-collection: hand the dataset to an estimator and ask for any observables.
Each `ShadowEstimate.error` is the half-width $\varepsilon=\sqrt{34\,\hat V/N}$ that
**Theorem 1** of Huang–Kueng–Preskill controls — a rigorous $1-\delta$ bound
($\hat V$ the empirical single-setting variance, $N$ the per-batch size). Its
constant $34$ is a deliberately loose worst case (the paper's own remark), so the
*true* error is typically far inside the bar: below you'll see accurate values
carrying wide guarantees.

`expval_many` makes the whole family **simultaneously valid** at level `delta` —
the batch count uses the union bound $K=\lceil 2\ln(2M/\delta)\rceil$, so all $M$
estimates lie within their bars jointly with probability $\ge 1-\delta$.


In [ ]:
est = shadow.dataset.estimator()
for label in ['Z1 Z2', 'Z2 Z3', 'X0 X1 X2 X3']:
    r = est.expval(label)
    print(f'<{label:9s}> = {r.value:+.3f} +/- {r.error:.3f}')

family = [QubitOperator('Z0 Z1'), QubitOperator('Z2 Z3')]
joint = est.expval_many(family)     # union-bounded batch count
print('joint n_batches =', joint[0].n_batches)


## Persist and reuse a campaign

The dataset is inert (no circuits) and serializes to NPZ; reload it later and
estimate a *new* operator without re-running the quantum circuits.


In [ ]:
import tempfile, os
path = os.path.join(tempfile.mkdtemp(), 'campaign.npz')
shadow.dataset.save(path)
loaded = ShadowDataset.load(path)
reuse = PauliShadow(QubitOperator('Z1 Z2'), dataset=loaded)
engine.build([reuse])
print('reuse <Z1 Z2> =', round(engine.run()[0], 3))


## Deterministic check (exact enumeration)

The sampled campaigns above are illustrative. The notebook's one **hard
assertion** is deterministic and tolerance-free: the shadow inverse channel is
*unbiased*, so averaged over **all** $3^n$ Pauli settings with their exact Born
weights it recovers $\langle H\rangle$ **exactly** (to machine precision) — this
is the identity the random campaigns converge toward. The `PauliAveraging`
cross-check below is shown, not asserted.


In [ ]:
import itertools
from qarp.algorithms import PauliKernel

bell = SimpleBlock(2)
bell.h(0)
bell.cx(0, 1)
bell.build()
H = QubitOperator('Z0 Z1') + 0.5 * QubitOperator('X0 X1') - 0.3 * QubitOperator('Z0')

# Enumerate every setting with its exact Born weights (no sampling). The per-axis
# basis change measured in Z is X->H, Y->H·S†, Z->I; kron with qubit 0 as the LSB.
psi = np.asarray(bell.statevector())
H1 = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
SDG = np.array([[1, 0], [0, -1j]], dtype=complex)
U_AXIS = {0: H1, 1: H1 @ SDG, 2: np.eye(2, dtype=complex)}


def kron_lsb(mats):
    full = mats[-1]
    for m in reversed(mats[:-1]):
        full = np.kron(full, m)
    return full


records = []
for axes in itertools.product((0, 1, 2), repeat=2):
    probs = np.abs(kron_lsb([U_AXIS[a] for a in axes]) @ psi) ** 2
    records.append((np.array(axes, dtype=np.int8),
                    {b: float(probs[b]) for b in range(4) if probs[b] > 1e-15}))
enumerated = ShadowDataset(PauliKernel(2), 2, records, shot_exact=True)
shadow_H = enumerated.estimator().expval(H, n_batches=1).value  # mean over all settings

# The oracle is analytic, not another qarp path: on the Bell state <Z0 Z1> = 1,
# <X0 X1> = 1 and <Z0> = 0, so <H> = 1 + 0.5*1 - 0.3*0 = 1.5 exactly.
EXACT_H = 1.5

# PauliAveraging is a *cross-check*, not the oracle -- it is another qarp path.
exact = PauliAveraging(ket=bell, operator=H, n_shots=qarp.EXACT)
eng2 = QarpEngine(seed=0)
eng2.build([exact])
exact_H = eng2.run()[0]

print(f'enumerated shadow <H> = {shadow_H:.12f}')
print(f'analytic       <H> = {EXACT_H:.12f}   (the oracle)')
print(f'PauliAveraging <H> = {exact_H:.12f}   (cross-check, another qarp path)')
assert abs(shadow_H - EXACT_H) < 1e-12  # unbiased inverse channel: exact, no tolerance
print('OK (exact to machine precision)')


## Convergence (shown, not asserted)

A *single* campaign's error is itself a random variable, so one seed per size
need not decrease monotonically — a small campaign can get lucky and a large one
unlucky. Averaging $|{\rm err}|$ over independent campaigns shows the actual
rate. For a weight-2 Pauli the single-setting variance is $3^2-1=8$, so the
error falls as $\sqrt{8/n_{\rm settings}}$ — the shadow norm setting the rate.


In [ ]:
R = 16  # independent campaigns per size (seeded, so the whole cell is reproducible)
for m in [500, 2000, 8000]:
    errs = []
    for s in range(R):
        sm = PauliShadow(QubitOperator('Z0 Z1'), bell, n_settings=m, seed=s)
        e = QarpEngine(seed=ENGINE_SEED + s)
        e.build([sm])
        errs.append(abs(e.run()[0] - 1))
    print(f'n_settings={m:5d}  mean |err| over {R} campaigns = {np.mean(errs):.4f}'
          f'   (sqrt(8/n_settings) = {np.sqrt(8 / m):.4f})')
